# 🎙️ Speech Emotion & Stress Detection

**Dataset:** [RAVDESS](https://zenodo.org/record/1188976) — 24 actors, 8 emotions, 1440 audio files  
**Pipeline:** Feature extraction → StandardScaler → SVM / Random Forest / Gradient Boosting  
**Features:** 40 MFCCs + delta + delta² + Chroma + Spectral Contrast + Mel Spectrogram + ZCR + RMS  

---

### What this notebook covers
1. Dataset overview and audio visualisation
2. Feature extraction explanation
3. Model training with cross-validation
4. Confusion matrix and ROC curves
5. Feature importance
6. Single-file inference

In [ ]:
# Install dependencies if running in Colab
# !pip install librosa scikit-learn matplotlib seaborn gradio joblib

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import our modules
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

from features import extract_features, load_dataset, INT_TO_EMOTION, EMOTION_MAP
from model import train_sklearn
from evaluate import plot_confusion_matrix, plot_roc_curves, plot_feature_importance, visualise_audio

from sklearn.model_selection import train_test_split

print('All imports OK')

## 1. Dataset Overview

RAVDESS filenames encode metadata:
```
03-01-05-01-02-02-12.wav
│  │  │  │  │  │  └── Actor ID
│  │  │  │  │  └───── Repetition
│  │  │  │  └──────── Statement
│  │  │  └─────────── Intensity
│  │  └────────────── Emotion (01=neutral … 08=surprised)
│  └───────────────── Statement type
└──────────────────── Modality
```

In [ ]:
# ── Update this path to point to your RAVDESS folder ──────────────────────
DATASET_PATH = '../data/RAVDESS'   # change this
# ──────────────────────────────────────────────────────────────────────────

# Count files per emotion
import os
from collections import Counter

from features import get_emotion_from_filename

emotion_counts = Counter()
for root, _, files in os.walk(DATASET_PATH):
    for f in files:
        if f.endswith('.wav'):
            emotion_counts[get_emotion_from_filename(f)] += 1

fig, ax = plt.subplots(figsize=(10, 4))
emotions = list(emotion_counts.keys())
counts   = list(emotion_counts.values())
bars = ax.bar(emotions, counts, color='steelblue', edgecolor='white')
ax.bar_label(bars)
ax.set_title('Samples per Emotion Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Total files: {sum(counts)}')

## 2. Audio Visualisation

Before extracting numbers, let's *see* what different emotions look like.

In [ ]:
# Replace with a real file path from your dataset
example_file = '../data/RAVDESS/Actor_01/03-01-05-01-02-02-12.wav'   # 'angry'

visualise_audio(example_file, save_path='../results/audio_viz_example.png')
from IPython.display import Image
Image('../results/audio_viz_example.png')

## 3. Feature Extraction

We extract **182 features** per audio clip:

| Feature group | Dim | What it captures |
|---|---|---|
| MFCCs (40 coeff) | 40 | Vocal tract shape / tone |
| Delta-MFCCs | 40 | Rate of change of tone |
| Delta²-MFCCs | 40 | Acceleration of change |
| Chroma | 12 | Pitch class energy |
| Spectral Contrast | 7 | Peak vs valley energy |
| Mel Spectrogram | 40 | Perceptual frequency energy |
| ZCR mean | 1 | Noisiness / consonant density |
| RMS mean + std | 2 | Loudness and its variation |

In [ ]:
# Test feature extraction on one file
feat = extract_features(example_file)
print(f'Feature vector shape: {feat.shape}')
print(f'Min: {feat.min():.3f}  Max: {feat.max():.3f}  Mean: {feat.mean():.3f}')

## 4. Load Full Dataset

In [ ]:
# First run: extracts features and caches to data/
# Subsequent runs: loads from cache instantly
X, y = load_dataset(DATASET_PATH, cache_dir='../data')

print(f'X shape: {X.shape}')   # (n_samples, 182)
print(f'y shape: {y.shape}')   # (n_samples,)
print(f'Classes: {np.unique(y, return_counts=True)}')

In [ ]:
# Stratified split — preserves class proportions in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

## 5. Train Model with Cross-Validation

In [ ]:
results = train_sklearn(
    X_train, y_train,
    X_test,  y_test,
    model_type='svm',          # try 'rf' or 'gb' too
    cv_folds=5,
    save_path='../models/svm_model.pkl',
)

cv = results['cv_scores']
print(f"\n5-Fold CV: {cv.mean():.3f} ± {cv.std():.3f}")

## 6. Evaluation

In [ ]:
model = results['model']
y_pred = model.predict(X_test)

class_names = [INT_TO_EMOTION[i] for i in range(len(INT_TO_EMOTION))]

plot_confusion_matrix(y_test, y_pred, class_names=class_names, save_path='../results/confusion_matrix.png')
from IPython.display import Image
Image('../results/confusion_matrix.png')

In [ ]:
plot_roc_curves(model, X_test, y_test, class_names=class_names, save_path='../results/roc_curves.png')
Image('../results/roc_curves.png')

## 7. Single File Inference

In [ ]:
def predict_file(file_path: str, model) -> None:
    feat = extract_features(file_path).reshape(1, -1)
    proba = model.predict_proba(feat)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = INT_TO_EMOTION[pred_idx]
    
    print(f"File: {Path(file_path).name}")
    print(f"Predicted: {pred_label.upper()} ({proba[pred_idx]*100:.1f}% confidence)\n")
    
    for i, (label, prob) in enumerate(zip(class_names, proba)):
        bar = '█' * int(prob * 40)
        marker = ' ← predicted' if i == pred_idx else ''
        print(f"{label:10s} {bar:<40s} {prob*100:5.1f}%{marker}")

predict_file(example_file, model)